# 05 — Node2Vec Game Embeddings

In this notebook, we use Node2Vec to learn vector representations of games from the structure of the behavioural similarity graph.

Node2Vec generates biased random walks through the graph and uses these walks to learn embeddings where games with similar graph neighbourhoods are represented by similar vectors.

Unlike Louvain, which assigns each game to a single community, Node2Vec provides a continuous representation that can capture more fine-grained relationships between games.

In [17]:
from pathlib import Path

import networkx as nx
import pandas as pd


GRAPH_PATH = Path("../data/processed/game_graph_edges.parquet")

edges = pd.read_parquet(GRAPH_PATH)

G = nx.from_pandas_edgelist(
    edges,
    source="game_a",
    target="game_b",
    edge_attr="jaccard_similarity"
)

print("Games:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Games: 2363
Edges: 18073


## Weighted random walks

Node2Vec learns from sequences of nodes generated by random walks through the graph.

We first implement a simple weighted random walk, where the probability of moving to a neighbouring game is proportional to the Jaccard similarity of the connecting edge. Stronger behavioural similarities are therefore more likely to be followed during the walk.

In the next step, we extend this with the Node2Vec return and exploration parameters.

In [18]:
import numpy as np


rng = np.random.default_rng(seed=42)


def weighted_random_walk(graph, start_node, walk_length):
    """Generate a weighted random walk from a starting node."""
    walk = [start_node]

    while len(walk) < walk_length:
        current_node = walk[-1]
        neighbours = list(graph.neighbors(current_node))

        if not neighbours:
            break

        weights = np.array([
            graph[current_node][neighbour]["jaccard_similarity"]
            for neighbour in neighbours
        ])

        probabilities = weights / weights.sum()

        next_node = rng.choice(
            neighbours,
            p=probabilities
        )

        walk.append(next_node)

    return walk

In [19]:
start_game = list(G.nodes())[0]

walk = weighted_random_walk(
    G,
    start_node=start_game,
    walk_length=10
)

print("Start game:", start_game)
print("Walk:", walk)
print("Walk length:", len(walk))

Start game: 467000
Walk: ['467000', np.str_('340650'), np.str_('463210'), np.str_('317250'), np.str_('434420'), np.str_('463210'), np.str_('323060'), np.str_('343710'), np.str_('248970'), np.str_('328500')]
Walk length: 10


## Node2Vec biased random walks

Node2Vec extends a standard random walk by considering both the current node and the node visited immediately before it.

The parameter `p` controls the probability of returning to the previous node, while `q` controls whether the walk tends to remain within the local neighbourhood or explore further through the graph.

The transition probabilities are also multiplied by the Jaccard edge weights, so stronger game similarities remain more likely to be followed.

In [20]:
def node2vec_walk(graph, start_node, walk_length, p=1.0, q=1.0):
    """Generate one Node2Vec biased random walk."""
    walk = [start_node]

    while len(walk) < walk_length:
        current_node = walk[-1]
        neighbours = list(graph.neighbors(current_node))

        if not neighbours:
            break

        weights = []

        for neighbour in neighbours:
            edge_weight = graph[current_node][neighbour]["jaccard_similarity"]

            # First step has no previous node.
            if len(walk) == 1:
                bias = 1.0

            else:
                previous_node = walk[-2]

                if neighbour == previous_node:
                    bias = 1.0 / p

                elif graph.has_edge(previous_node, neighbour):
                    bias = 1.0

                else:
                    bias = 1.0 / q

            weights.append(edge_weight * bias)

        probabilities = np.array(weights) / np.sum(weights)

        next_node = rng.choice(
            neighbours,
            p=probabilities
        )

        walk.append(str(next_node))

    return walk

In [21]:
walk = node2vec_walk(
    G,
    start_node=start_game,
    walk_length=10,
    p=1.0,
    q=1.0
)

print("Start game:", start_game)
print("Walk:", walk)
print("Walk length:", len(walk))

Start game: 467000
Walk: ['467000', '423710', '412170', '463350', '434420', '391420', '355150', '306040', '366890', '396160']
Walk length: 10


In [22]:
settings = [
    ("Balanced", 1.0, 1.0),
    ("More local", 1.0, 2.0),
    ("More exploratory", 1.0, 0.5),
]

for name, p, q in settings:
    walk = node2vec_walk(
        G,
        start_node=start_game,
        walk_length=10,
        p=p,
        q=q
    )

    print(f"\n{name} walk (p={p}, q={q})")
    print(walk)


Balanced walk (p=1.0, q=1.0)
['467000', '454890', '428950', '438180', '403560', '232770', '234940', '258200', '338050', '234940']

More local walk (p=1.0, q=2.0)
['467000', '457820', '448640', '340650', '423710', '463350', '463210', '454890', '438030', '423710']

More exploratory walk (p=1.0, q=0.5)
['467000', '457820', '385070', '448640', '385070', '331790', '385080', '265120', '375620', '251910']


## Generate Node2Vec walks

We use `p = 1` and `q = 2`, giving the random walks a moderate preference for staying within local graph neighbourhoods.

For each game, we generate multiple walks. These sequences will later be used as the training data for Word2Vec, treating games as words and random walks as sentences.

In [23]:
def generate_walks(
    graph,
    num_walks=10,
    walk_length=20,
    p=1.0,
    q=2.0
):
    """Generate multiple Node2Vec walks starting from every node."""
    walks = []
    nodes = list(graph.nodes())

    for _ in range(num_walks):
        rng.shuffle(nodes)

        for node in nodes:
            walk = node2vec_walk(
                graph,
                start_node=node,
                walk_length=walk_length,
                p=p,
                q=q
            )

            walks.append(walk)

    return walks

In [24]:
walks = generate_walks(
    G,
    num_walks=10,
    walk_length=20,
    p=1.0,
    q=2.0
)

print("Number of walks:", len(walks))
print("Example walk:", walks[0])
print("Example walk length:", len(walks[0]))

Number of walks: 23630
Example walk: ['362890', '380', '7670', '24980', '17460', '22380', '413150', '377160', '211420', '374320', '211420', '200710', '238960', '211420', '335300', '367500', '235540', '394510', '56400', '364360']
Example walk length: 20


## Learn game embeddings with Word2Vec

The generated Node2Vec walks can be treated similarly to sentences, where each game is a word and games appearing in similar walk contexts should receive similar vector representations.

We use Word2Vec with the skip-gram objective to learn these embeddings. The random walks are generated by our own Node2Vec implementation, while Word2Vec itself is provided by the `gensim` library.

In [25]:
from gensim.models import Word2Vec


model = Word2Vec(
    sentences=walks,
    vector_size=64,
    window=5,
    min_count=1,
    sg=1,
    workers=1,
    epochs=10,
    seed=42
)

print("Games with embeddings:", len(model.wv))
print("Embedding dimensions:", model.wv.vector_size)

Games with embeddings: 2363
Embedding dimensions: 64


## Inspect learned game similarities

We now inspect the nearest games in the learned embedding space. If Node2Vec captures meaningful graph structure, games with similar player audiences and graph neighbourhoods should have similar embedding vectors.

In [26]:
# Load game names.
interactions = pd.read_parquet(
    "../data/processed/interactions_filtered.parquet"
)

game_names = (
    interactions[["game_id", "game_name"]]
    .drop_duplicates(subset=["game_id"])
)

game_id_to_name = dict(
    zip(game_names["game_id"], game_names["game_name"])
)

game_name_to_id = dict(
    zip(game_names["game_name"], game_names["game_id"])
)


def most_similar_games(game_name, topn=10):
    """Return the most similar games in the Node2Vec embedding space."""
    game_id = game_name_to_id[game_name]

    similar_games = model.wv.most_similar(
        str(game_id),
        topn=topn
    )

    results = []

    for similar_id, similarity in similar_games:
        results.append({
            "game_name": game_id_to_name.get(similar_id, similar_id),
            "similarity": similarity
        })

    return pd.DataFrame(results)

In [27]:
most_similar_games(
    "The Elder Scrolls V: Skyrim"
)

,game_name,similarity
0,Borderlands 2,0.853432
1,Left 4 Dead 2,0.833091
2,Terraria,0.828938
3,Sid Meier's Civilization V,0.828335
4,Counter-Strike: Global Offensive,0.810737
5,Garry's Mod,0.808960
6,Ace of Spades,0.808121
7,PAYDAY 2,0.797019
8,Chivalry: Medieval Warfare,0.782956
9,Portal 2,0.780173


### Example: Skyrim

The nearest games to *The Elder Scrolls V: Skyrim* include titles such as *Borderlands 2*, *Terraria*, *Civilization V*, *PAYDAY 2*, and *Portal 2*.

These games do not all belong to the same genre, suggesting that the Node2Vec embeddings capture broader similarities in player behaviour and graph neighbourhood structure rather than simply reproducing genre categories.

In [28]:
most_similar_games(
    "Total War: SHOGUN 2"
)

,game_name,similarity
0,Total War: ROME II - Emperor Edition,0.898588
1,Empire: Total War,0.842504
2,Medieval II: Total War,0.835329
3,Total War: ATTILA,0.824977
4,Napoleon: Total War,0.822065
5,Mount & Blade: Warband,0.817203
6,Total War: WARHAMMER,0.816760
7,Rome: Total War,0.810425
8,Crusader Kings II,0.789387
9,Company of Heroes (New Steam Version),0.785216


### Example: Total War: SHOGUN 2

The nearest games to *Total War: SHOGUN 2* are dominated by other *Total War* titles, alongside related strategy games such as *Crusader Kings II* and *Company of Heroes*.

This provides an even stronger sanity check than the Skyrim example, because the embedding recovers a very coherent strategy-game neighbourhood purely from the structure of the behavioural similarity graph.

## Quantitative evaluation of the embeddings

To evaluate whether Node2Vec preserves meaningful relationships from the behavioural similarity graph, we compare each game's nearest neighbours in the embedding space with its strongest neighbours in the original graph.

For each game, we take the 10 neighbours with the highest Jaccard edge weights and compare them with the 10 most similar games returned by Node2Vec. We then calculate the average neighbour overlap across all games.

In [29]:
def strongest_graph_neighbours(graph, node, topn=10):
    """Return the strongest graph neighbours based on Jaccard similarity."""
    neighbours = []

    for neighbour in graph.neighbors(node):
        similarity = graph[node][neighbour]["jaccard_similarity"]
        neighbours.append((str(neighbour), similarity))

    neighbours.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        neighbour
        for neighbour, _ in neighbours[:topn]
    ]


overlaps = []

for node in G.nodes():
    node = str(node)

    graph_neighbours = set(
        strongest_graph_neighbours(
            G,
            node,
            topn=10
        )
    )

    embedding_neighbours = set(
        game_id
        for game_id, _ in model.wv.most_similar(
            node,
            topn=10
        )
    )

    overlap = len(
        graph_neighbours & embedding_neighbours
    )

    overlaps.append(
        overlap / len(graph_neighbours)
    )


average_overlap = np.mean(overlaps)

print(
    "Average top-10 neighbour overlap:",
    f"{average_overlap:.2%}"
)

Average top-10 neighbour overlap: 58.47%


In [30]:
random_overlaps = []
nodes = [str(node) for node in G.nodes()]

for node in nodes:
    graph_neighbours = set(
        strongest_graph_neighbours(
            G,
            node,
            topn=10
        )
    )

    possible_nodes = [
        other
        for other in nodes
        if other != node
    ]

    random_neighbours = set(
        rng.choice(
            possible_nodes,
            size=10,
            replace=False
        )
    )

    overlap = len(
        graph_neighbours & random_neighbours
    )

    random_overlaps.append(
        overlap / len(graph_neighbours)
    )

print(
    "Node2Vec top-10 overlap:",
    f"{average_overlap:.2%}"
)

print(
    "Random top-10 overlap:",
    f"{np.mean(random_overlaps):.2%}"
)

Node2Vec top-10 overlap: 58.47%
Random top-10 overlap: 0.45%


### Embedding evaluation

The average overlap between the 10 strongest Jaccard neighbours and the 10 nearest Node2Vec neighbours is **58.47%**.

For comparison, a random set of 10 games produces an average overlap of only **0.45%**. This shows that the learned embeddings preserve a substantial amount of meaningful local graph structure.

The overlap is not expected to be 100%, since Node2Vec also captures broader neighbourhood relationships through its random walks rather than simply reproducing direct graph edges.

## Recommendation evaluation

As a practical application of the learned Node2Vec embeddings, we evaluate whether they can be used to recommend games to individual users.

For each test user, one engaged game is hidden from their history. The remaining game embeddings are averaged to create a user profile vector, which is then used to rank games the user has not already played.

We evaluate whether the hidden game can be recovered among the highest-ranked recommendations.

In [31]:
# Games for which we have learned Node2Vec embeddings.
embedded_games = set(model.wv.index_to_key)

# Keep only interactions with games represented in the embedding space.
recommendation_interactions = interactions[
    interactions["game_id"].isin(embedded_games)
].copy()

# Create a list of engaged games for each user.
user_games = (
    recommendation_interactions
    .groupby("user_id")["game_id"]
    .apply(list)
)

# Require at least 5 games so that the user profile still contains
# several games after hiding one.
eligible_users = user_games[
    user_games.apply(len) >= 5
]

print("Users with embedded games:", len(user_games))
print("Users eligible for evaluation:", len(eligible_users))

print("\nGames for example user:")
print(eligible_users.iloc[0])

Users with embedded games: 66746
Users eligible for evaluation: 55091

Games for example user:
['730', '339610', '304050', '304930', '1250', '113400', '550', '55230', '242720', '204360', '105600', '4000', '24240', '265630', '238460', '104700', '302830', '218620', '263760', '233190', '200210', '270550', '9480']


### Held-out game recommendation

We evaluate the embeddings by randomly hiding one engaged game from each test user's history. The remaining game embeddings are averaged to form a user profile, and all games not already present in the user's remaining history are ranked by cosine similarity to this profile.

We report Hit Rate@10 and Hit Rate@20, measuring how often the hidden game appears among the top 10 or top 20 recommendations.

Since the embeddings were trained on the full interaction graph, this experiment should be interpreted as a practical recommendation sanity check rather than a fully out-of-sample evaluation.

In [32]:
# Evaluate on a reproducible sample to keep the experiment fast.
NUM_TEST_USERS = 1000
eval_rng = np.random.default_rng(seed=42)

test_users = eval_rng.choice(
    eligible_users.index,
    size=NUM_TEST_USERS,
    replace=False
)

# Normalized embedding matrix for cosine similarity.
embedding_ids = model.wv.index_to_key
embedding_matrix = model.wv.get_normed_vectors()

game_to_index = {
    game_id: index
    for index, game_id in enumerate(embedding_ids)
}

ranks = []

for user_id in test_users:
    games = eligible_users.loc[user_id]

    # Randomly hide one engaged game.
    hidden_game = eval_rng.choice(games)

    known_games = [
        game
        for game in games
        if game != hidden_game
    ]

    # Average the remaining game embeddings into a user profile.
    profile_vectors = np.array([
        model.wv[game]
        for game in known_games
    ])

    user_profile = profile_vectors.mean(axis=0)
    user_profile = user_profile / np.linalg.norm(user_profile)

    # Cosine similarity between the user profile and every game.
    scores = embedding_matrix @ user_profile

    # Do not recommend games already known by the user.
    for game in known_games:
        scores[game_to_index[game]] = -np.inf

    hidden_index = game_to_index[hidden_game]
    hidden_score = scores[hidden_index]

    # Rank of the hidden game among unseen games.
    rank = 1 + np.sum(scores > hidden_score)
    ranks.append(rank)


ranks = np.array(ranks)

hit_rate_10 = np.mean(ranks <= 10)
hit_rate_20 = np.mean(ranks <= 20)
median_rank = np.median(ranks)

print("Test users:", NUM_TEST_USERS)
print("Hit Rate @ 10:", f"{hit_rate_10:.2%}")
print("Hit Rate @ 20:", f"{hit_rate_20:.2%}")
print("Median hidden-game rank:", int(median_rank))

Test users: 1000
Hit Rate @ 10: 27.10%
Hit Rate @ 20: 38.10%
Median hidden-game rank: 38


### Popularity baseline

To provide context for the Node2Vec recommendation results, we compare against a simple popularity-based recommender.

The baseline ranks unseen games by the number of engaged users in the dataset, without using any information about the individual user's preferences. We use the same users and held-out games as in the Node2Vec evaluation.

In [33]:
# Global popularity of each game based on engaged users.
game_popularity = (
    recommendation_interactions
    .groupby("game_id")["user_id"]
    .nunique()
)

popularity_scores = np.array([
    game_popularity.get(game_id, 0)
    for game_id in embedding_ids
], dtype=float)


# Recreate the exact same test users and hidden games.
baseline_rng = np.random.default_rng(seed=42)

baseline_test_users = baseline_rng.choice(
    eligible_users.index,
    size=NUM_TEST_USERS,
    replace=False
)

popularity_ranks = []

for user_id in baseline_test_users:
    games = eligible_users.loc[user_id]

    hidden_game = baseline_rng.choice(games)

    known_games = [
        game
        for game in games
        if game != hidden_game
    ]

    scores = popularity_scores.copy()

    # Do not recommend games already known by the user.
    for game in known_games:
        scores[game_to_index[game]] = -np.inf

    hidden_index = game_to_index[hidden_game]
    hidden_score = scores[hidden_index]

    rank = 1 + np.sum(scores > hidden_score)
    popularity_ranks.append(rank)


popularity_ranks = np.array(popularity_ranks)

popularity_hit_10 = np.mean(popularity_ranks <= 10)
popularity_hit_20 = np.mean(popularity_ranks <= 20)
popularity_median_rank = np.median(popularity_ranks)

print("Node2Vec")
print("Hit Rate @ 10:", f"{hit_rate_10:.2%}")
print("Hit Rate @ 20:", f"{hit_rate_20:.2%}")
print("Median rank:", int(median_rank))

print("\nPopularity baseline")
print("Hit Rate @ 10:", f"{popularity_hit_10:.2%}")
print("Hit Rate @ 20:", f"{popularity_hit_20:.2%}")
print("Median rank:", int(popularity_median_rank))

Node2Vec
Hit Rate @ 10: 27.10%
Hit Rate @ 20: 38.10%
Median rank: 38

Popularity baseline
Hit Rate @ 10: 25.00%
Hit Rate @ 20: 33.20%
Median rank: 63


### Recommendation results

The Node2Vec recommender outperforms the popularity baseline across all evaluation metrics. The hidden game is recovered within the top 10 recommendations for 27.1% of users and within the top 20 for 38.1%, compared with 25.0% and 33.2% for the popularity baseline.

The median rank of the hidden game is also substantially better with Node2Vec: 38 compared with 63 for the popularity baseline. This suggests that the learned graph embeddings capture personalized behavioural structure beyond simply recommending globally popular games.

The improvement is moderate rather than dramatic, but it provides evidence that the graph-based representation is useful for recommendation.

In [34]:
def recommend_games(user_id, topn=10):
    """Recommend games to a user from their Node2Vec profile."""

    games = eligible_users.loc[user_id]

    profile_vectors = np.array([
        model.wv[game]
        for game in games
    ])

    user_profile = profile_vectors.mean(axis=0)
    user_profile = user_profile / np.linalg.norm(user_profile)

    scores = embedding_matrix @ user_profile

    # Do not recommend games the user has already played.
    for game in games:
        scores[game_to_index[game]] = -np.inf

    top_indices = np.argsort(scores)[::-1][:topn]

    recommendations = []

    for index in top_indices:
        game_id = embedding_ids[index]

        recommendations.append({
            "game": game_id_to_name.get(game_id, game_id),
            "similarity": scores[index]
        })

    return pd.DataFrame(recommendations)

In [35]:
example_user = eligible_users.index[0]

print("Games already played:")
for game_id in eligible_users.loc[example_user]:
    print("-", game_id_to_name.get(game_id, game_id))

print("\nRecommendations:")
recommend_games(example_user, topn=10)

Games already played:
- Counter-Strike: Global Offensive
- FreeStyle2: Street Basketball
- Trove
- Unturned
- Killing Floor
- APB Reloaded
- Left 4 Dead 2
- Saints Row: The Third
- GunZ 2: The Second Duel
- Castle Crashers
- Terraria
- Garry's Mod
- PAYDAY: The Heist
- Fistful of Frags
- BattleBlock Theater
- Super Monday Night Combat
- BLOCKADE 3D
- PAYDAY 2
- Turbo Dismount
- Final Exam
- Realm of the Mad God
- Yet Another Zombie Defense
- Saints Row 2

Recommendations:


,game,similarity
0,Loadout,0.897270
1,Warframe,0.893214
2,Robocraft,0.858069
3,No More Room in Hell,0.851432
4,Toribash,0.850844
5,War Thunder,0.848900
6,Tom Clancy's Ghost Recon Phantoms - NA,0.848812
7,Rust,0.848617
8,PlanetSide 2,0.845353
9,Paladins,0.841278


### Recommendation example

The recommendation example shows that the learned Node2Vec embeddings can be used to construct personalized game recommendations from a user's existing play history.

For a user with a strong preference for multiplayer and action-oriented games, the recommender suggests games such as *Loadout*, *Warframe*, *Robocraft*, *War Thunder*, *Rust*, and *PlanetSide 2*. These recommendations are consistent with the behavioural patterns visible in the user's previously played games.

Together with the held-out evaluation, where Node2Vec achieved a Hit Rate@10 of 27.1% compared with 25.0% for the popularity baseline, this demonstrates a practical use of the learned graph embeddings.